In [ ]:
# here you can apply the post-processing pipeline developed in the thesis
# this was developed as a way to mitigate an imperfect model

# the main goal of this pipeline is to remove likely false detections of crab
# whilst keeping most of the true detections without setting a high confidence threshold
#

In [ ]:
# Chunk 1 - imports and confid
import json
import math
from pathlib import Path
from collections import defaultdict

import cv2
import numpy as np
import pandas as pd


def find_repo_root(start=None, marker=".git"):
    path = Path(start or Path.cwd()).resolve()
    for parent in [path, *path.parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"Could not find a repo root (looking for '{marker}')")


REPO_ROOT = find_repo_root()

CONFIG = {
    "PREDICTIONS": REPO_ROOT / "use-a-crab-detector" / "data" / "predictions" / "predictions.json",
    "VIDEOS_DIR": REPO_ROOT / "use-a-crab-detector" / "data" / "videos",
    "OUTPUT_DIR": REPO_ROOT / "use-a-crab-detector" / "data" / "predictions" / "filtered",
    "DEDUP_IOU": 0.3,           # boxes overlapping more than this are treated as the same crab
    "CONF_THRESHOLD": 0.3,      # discard detections below this confidence before tracking
    "TRACK_IOU_THRESH": 0.3,
    "TRACK_DIST_THRESH_PX": 60,     # how close two boxes' centers must be (pixels) to link across frames; tune to your video resolution
    "MAX_FRAME_GAP": 2,             # allow a track to skip this many frames without a matching detection
    "MAX_MISSED": 2,
    "MIN_TRACK_LEN": 3,              # tracks shorter than this are likely flickers, not real crabs
    "MAX_STATIC_LEN": 500,           # tracks longer than this that barely move are likely a fixed object, not a crab
    "STATIC_MAX_DISP_PX": 15,
}
CONFIG["OUTPUT_DIR"].mkdir(parents=True, exist_ok=True)

In [ ]:
# Chunk 1 - load predictions into a per-video, per-frame index
def build_prediction_index(predictions_path: Path) -> dict:
    """
    Load predictions.json and group detections by video and frame number,
    parsed from the image_id (format: "<video_stem>_frame_<n>").
    """
    predictions = json.loads(predictions_path.read_text())
    index = defaultdict(lambda: defaultdict(list))

    for pred in predictions:
        video_id, frame_str = pred["image_id"].rsplit("_frame_", 1)
        frame = int(frame_str)
        x, y, w, h = pred["bbox"]
        index[video_id][frame].append({
            "video_id": video_id,
            "frame": frame,
            "bbox": (x, y, x + w, y + h),  # convert to (x1, y1, x2, y2) for IoU math
            "conf": pred["score"],
        })

    return index


pred_index = build_prediction_index(CONFIG["PREDICTIONS"])

In [ ]:
# Chunk 2 - geometry helpers

def compute_iou(b1, b2) -> float:
    x1, y1 = max(b1[0], b2[0]), max(b1[1], b2[1])
    x2, y2 = min(b1[2], b2[2]), min(b1[3], b2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    if inter == 0:
        return 0.0
    a1 = (b1[2] - b1[0]) * (b1[3] - b1[1])
    a2 = (b2[2] - b2[0]) * (b2[3] - b2[1])
    return inter / (a1 + a2 - inter)


def box_center(b):
    return ((b[0] + b[2]) / 2, (b[1] + b[3]) / 2)


def center_dist(b1, b2) -> float:
    c1, c2 = box_center(b1), box_center(b2)
    return math.hypot(c1[0] - c2[0], c1[1] - c2[1])

In [ ]:
# Chunk 3 - remove duplicate boxes on the same crab within a frame

def deduplicate_boxes(boxes: list, iou_thresh: float) -> list:
    """When the model draws two overlapping boxes for one crab, keep only the larger one."""
    if len(boxes) <= 1:
        return boxes

    order = sorted(range(len(boxes)), key=lambda i: -(boxes[i]["bbox"][2] - boxes[i]["bbox"][0]) * (boxes[i]["bbox"][3] - boxes[i]["bbox"][1]))
    kept, removed = [], set()
    for i in order:
        if i in removed:
            continue
        kept.append(boxes[i])
        for j in order:
            if j != i and j not in removed and compute_iou(boxes[i]["bbox"], boxes[j]["bbox"]) > iou_thresh:
                removed.add(j)
    return kept


def apply_to_index(index: dict, fn, *args) -> dict:
    """Apply a per-frame filtering function across every video and frame in an index."""
    result = defaultdict(lambda: defaultdict(list))
    for vid, frames in index.items():
        for f, boxes in frames.items():
            result[vid][f] = fn(boxes, *args)
    return result


pred_index_dedup = apply_to_index(pred_index, deduplicate_boxes, CONFIG["DEDUP_IOU"])

In [ ]:
# Chunk 5 - drop low confidence detections 

pred_index_conf = apply_to_index(
    pred_index_dedup,
    lambda boxes, thresh: [b for b in boxes if b["conf"] >= thresh],
    CONFIG["CONF_THRESHOLD"],
)

In [ ]:
# Chunk 5 - soft tracker: link detections of the same crab across frames

def build_soft_tracks(frames: dict, iou_thresh: float, dist_thresh: float, max_frame_gap: int, max_missed: int) -> list:
    """
    Link detections across consecutive frames into tracks, based on box
    overlap or how close the box centers are. This turns a pile of
    per-frame detections into a much more meaningful unit: one track per
    crab (or crab-like object) actually being watched over time.
    """
    tracks, finished, tid = [], [], 0

    for frame in sorted(frames):
        dets = frames[frame]
        assigned = set()

        for tr in tracks:
            if frame - tr["last_frame"] > max_frame_gap:
                tr["missed"] += 1
                continue
            best, best_score = None, -1
            for i, d in enumerate(dets):
                if i in assigned:
                    continue
                iou = compute_iou(tr["detections"][-1]["bbox"], d["bbox"])
                dist = center_dist(tr["detections"][-1]["bbox"], d["bbox"])
                if iou > iou_thresh or dist < dist_thresh:
                    score = iou - dist
                    if score > best_score:
                        best, best_score = i, score
            if best is not None:
                tr["detections"].append(dets[best])
                tr["last_frame"], tr["missed"] = frame, 0
                assigned.add(best)
            else:
                tr["missed"] += 1

        tracks = [tr for tr in tracks if tr["missed"] <= max_missed or finished.append(tr)]
        for i, d in enumerate(dets):
            if i not in assigned:
                tracks.append({"id": tid, "detections": [d], "last_frame": frame, "missed": 0})
                tid += 1

    finished.extend(tracks)
    return finished

In [ ]:
# Chunk 6 - remove flicker and static-object tracks

def track_is_static(track: dict, max_disp: float) -> bool:
    """A track whose box barely moves over its whole lifetime is likely a fixed object, not a crab."""
    centers = [box_center(d["bbox"]) for d in track["detections"]]
    xs, ys = [c[0] for c in centers], [c[1] for c in centers]
    movement = math.hypot(max(xs) - min(xs), max(ys) - min(ys))
    return movement < max_disp


def filter_tracks(pred_index: dict, config: dict) -> dict:
    """
    Build tracks per video, then discard tracks that are too short to be
    real (flickers, likely noise) or too long and too still (likely a
    static object such as a rock or the bait bag, not a moving crab).
    """
    kept_by_video = {}
    for vid, frames in pred_index.items():
        tracks = build_soft_tracks(
            frames, config["TRACK_IOU_THRESH"], config["TRACK_DIST_THRESH_PX"],
            config["MAX_FRAME_GAP"], config["MAX_MISSED"],
        )
        kept = []
        for tr in tracks:
            length = len(tr["detections"])
            if length < config["MIN_TRACK_LEN"]:
                continue
            if length > config["MAX_STATIC_LEN"] and track_is_static(tr, config["STATIC_MAX_DISP_PX"]):
                continue
            kept.append(tr)
        kept_by_video[vid] = kept
    return kept_by_video


filtered_tracks_by_video = filter_tracks(pred_index_conf, CONFIG)

# Rebuild a frame index from the surviving tracks, and dedup once more
# since two separate tracks can still briefly overlap.
final_pred_index = defaultdict(lambda: defaultdict(list))
for vid, tracks in filtered_tracks_by_video.items():
    frame_dets = defaultdict(list)
    for tr in tracks:
        for det in tr["detections"]:
            frame_dets[det["frame"]].append(det)
    for f, boxes in frame_dets.items():
        final_pred_index[vid][f] = deduplicate_boxes(boxes, CONFIG["DEDUP_IOU"])

In [ ]:
# Chunk 7 - per-station maxN raw vs filtered

STATION_PATTERN_LOOSE = __import__("re").compile(r"_([A-Z]{1,3}\d{1,3})_")
# Looser than the STATION_PATTERN used elsewhere in the repo, since this
# matches raw video filenames rather than already-extracted frame filenames.


def videos_by_station(video_ids) -> dict:
    grouped = defaultdict(list)
    for vid in video_ids:
        m = STATION_PATTERN_LOOSE.search(vid)
        if m:
            grouped[m.group(1)].append(vid)
    return grouped


def compute_maxn_by_station(pred_index: dict, station_videos: dict) -> dict:
    """For each station, find the single frame with the most crabs detected (maxN)."""
    results = {}
    for station, vids in station_videos.items():
        best = {"count": 0, "video_id": None, "frame": None}
        for vid in vids:
            for frame, dets in pred_index.get(vid, {}).items():
                if len(dets) > best["count"]:
                    best = {"count": len(dets), "video_id": vid, "frame": frame}
        results[station] = best
    return results


station_videos = videos_by_station(pred_index.keys())
raw_maxn = compute_maxn_by_station(pred_index, station_videos)
filtered_maxn = compute_maxn_by_station(final_pred_index, station_videos)

maxn_df = pd.DataFrame([
    {
        "station": s,
        "raw_maxN": raw_maxn[s]["count"], "raw_video": raw_maxn[s]["video_id"], "raw_frame": raw_maxn[s]["frame"],
        "filtered_maxN": filtered_maxn[s]["count"], "filtered_video": filtered_maxn[s]["video_id"], "filtered_frame": filtered_maxn[s]["frame"],
    }
    for s in sorted(raw_maxn)
])
maxn_df.to_csv(CONFIG["OUTPUT_DIR"] / "maxn_raw_vs_filtered.csv", index=False)
print(f"Saved maxn_raw_vs_filtered.csv ({len(maxn_df)} stations)")

In [ ]:
# Chunk 8 - pipeline summary: detections remaining at each stage

def count_detections(index: dict) -> int:
    return sum(len(boxes) for frames in index.values() for boxes in frames.values())


pipeline_summary = pd.DataFrame({
    "stage": ["Raw detections", "After deduplication", "After confidence filter", "After tracking filter"],
    "detections": [
        count_detections(pred_index),
        count_detections(pred_index_dedup),
        count_detections(pred_index_conf),
        count_detections(final_pred_index),
    ],
})
pipeline_summary["removed_from_previous"] = pipeline_summary["detections"].diff().fillna(0).abs()
pipeline_summary.to_csv(CONFIG["OUTPUT_DIR"] / "postprocessing_pipeline_summary.csv", index=False)
print(pipeline_summary)

In [ ]:
# Chunk 9 - visual before/after check (optional, need access to the orignial videos)

def find_video_from_id(video_id: str, videos_dir: Path) -> Path | None:
    matches = list(Path(videos_dir).rglob(f"{video_id}.*"))
    return matches[0] if matches else None


def draw_bbox(frame, bbox, conf=None, color=(0, 255, 0), thickness=3):
    x1, y1, x2, y2 = map(int, bbox)
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, thickness)
    if conf is not None:
        cv2.putText(frame, f"{conf:.2f}", (x1, max(y1 - 10, 20)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2, cv2.LINE_AA)


def save_maxn_frame(maxn_info: dict, pred_index: dict, videos_dir: Path, out_dir: Path, label: str, vid_stride: int = 30):
    """Save one annotated image per station showing its maxN frame, for a quick visual sanity check."""
    out_dir.mkdir(parents=True, exist_ok=True)
    for station, info in maxn_info.items():
        video_id, frame_idx = info["video_id"], info["frame"]
        if video_id is None:
            continue
        video_path = find_video_from_id(video_id, videos_dir)
        if video_path is None:
            print(f"Video not found for {station}: {video_id}")
            continue

        cap = cv2.VideoCapture(str(video_path))
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx * vid_stride)
        ok, frame = cap.read()
        cap.release()
        if not ok:
            print(f"Could not read frame for {station}")
            continue

        for d in pred_index[video_id].get(frame_idx, []):
            draw_bbox(frame, d["bbox"], d.get("conf"))
        cv2.putText(frame, f"{station} | {label} maxN={info['count']}", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.1, (0, 255, 0), 3, cv2.LINE_AA)
        cv2.imwrite(str(out_dir / f"{station}_{label.upper()}.jpg"), frame)

    print(f"Saved {label} maxN frames to {out_dir}")


save_maxn_frame(raw_maxn, pred_index, CONFIG["VIDEOS_DIR"], CONFIG["OUTPUT_DIR"] / "maxn_frames", "raw")
save_maxn_frame(filtered_maxn, final_pred_index, CONFIG["VIDEOS_DIR"], CONFIG["OUTPUT_DIR"] / "maxn_frames", "filtered")